# 0. les biblios

In [41]:
import pandas as pd
import re

# 1. Charger bat_surveys.csv

In [10]:
# Charger les données
df = pd.read_csv("bat_surveys.csv")

In [11]:
# 1. Dimensions et types de colonnes
print("Dimensions :", df.shape)
print("\nTypes de colonnes :")
df.info()

Dimensions : (4994, 14)

Types de colonnes :
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4994 entries, 0 to 4993
Data columns (total 14 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   survey_id                 4994 non-null   object 
 1   site_code                 4994 non-null   object 
 2   survey_start              4994 non-null   object 
 3   survey_end                4994 non-null   object 
 4   survey_method             4994 non-null   object 
 5   observation_hours         4994 non-null   object 
 6   observed_bat_count        4994 non-null   object 
 7   acoustic_detection_count  4994 non-null   object 
 8   nightly_visit_count       4994 non-null   object 
 9   colony_estimate           4994 non-null   float64
 10  comparison_species_count  4994 non-null   int64  
 11  observer_team             4974 non-null   object 
 12  survey_quality            4994 non-null   object 
 13  weather_interferen

In [12]:
# 2. Échantillon des données
print("\nPremières lignes :")
display(df.head())


Premières lignes :


,survey_id,site_code,survey_start,survey_end,survey_method,observation_hours,observed_bat_count,acoustic_detection_count,nightly_visit_count,colony_estimate,comparison_species_count,observer_team,survey_quality,weather_interference
0,BAT-0000001,KNT-KES,2019-01-05T18:00:00,2019-01-12T00:00:00,fixed_acoustic_station,28.6,7,13,7,3.8,8,TEAM-11-2,high,32.0
1,BAT-0000002,KNT-KES,2019-02-05T18:00:00,2019-02-12T00:00:00,acoustic_and_visual,31.7,8,12,9,7.3,18,TEAM-11-2,acceptable,18.5
2,BAT-0000003,KNT-KES,2019-03-05T18:00:00,2019-03-12T00:00:00,fixed_acoustic_station,32.4,22,28,14,19.9,8,TEAM-11-2,high,26.2
3,BAT-0000004,KNT-KES,2019-04-05T18:00:00,2019-04-12T00:00:00,acoustic_and_visual,35.7,26,36,19,32.2,14,TEAM-11-2,high,35.0
4,BAT-0000005,KNT-KES,2019-05-05T18:00:00,2019-05-12T00:00:00,acoustic_transect,32.0,23,29,22,20.9,16,TEAM-11-2,acceptable,38.7


In [13]:
# 3. Doublons
print("Nombre de lignes dupliquées (strictement identiques) :", df.duplicated().sum())
print("Nombre de survey_id dupliqués :", df['survey_id'].duplicated().sum())

# Afficher les doublons de survey_id s'il y en a
dupes = df[df['survey_id'].duplicated(keep=False)]
if not dupes.empty:
    display(dupes.sort_values('survey_id'))

Nombre de lignes dupliquées (strictement identiques) : 2
Nombre de survey_id dupliqués : 2


,survey_id,site_code,survey_start,survey_end,survey_method,observation_hours,observed_bat_count,acoustic_detection_count,nightly_visit_count,colony_estimate,comparison_species_count,observer_team,survey_quality,weather_interference
3182,BAT-0003183,AVL-ABG,2024-03-05T18:00:00,2024-03-12T00:00:00,acoustic_and_visual,38.7,8,18,12,8.6,7,TEAM-12-4,high,7.3
4992,BAT-0003183,AVL-ABG,2024-03-05T18:00:00,2024-03-12T00:00:00,acoustic_and_visual,38.7,8,18,12,8.6,7,TEAM-12-4,high,7.3
3566,BAT-0003567,AVL-FNQ,2023-09-05T18:00:00,2023-09-12T00:00:00,acoustic_transect,28.9,13,4,5,14.8,8,TEAM-12-4,high,55.9
4993,BAT-0003567,AVL-FNQ,2023-09-05T18:00:00,2023-09-12T00:00:00,acoustic_transect,28.9,13,4,5,14.8,8,TEAM-12-4,high,55.9


# 2. Valeurs manquantes

### a. Constat du problème

In [14]:
# 4. Valeurs manquantes
print("Valeurs manquantes par colonne :")
print(df.isna().sum())

print("\nPourcentage de manquants par colonne :")
print((df.isna().sum() / len(df) * 100).round(2))

Valeurs manquantes par colonne :
survey_id                    0
site_code                    0
survey_start                 0
survey_end                   0
survey_method                0
observation_hours            0
observed_bat_count           0
acoustic_detection_count     0
nightly_visit_count          0
colony_estimate              0
comparison_species_count     0
observer_team               20
survey_quality               0
weather_interference         0
dtype: int64

Pourcentage de manquants par colonne :
survey_id                   0.0
site_code                   0.0
survey_start                0.0
survey_end                  0.0
survey_method               0.0
observation_hours           0.0
observed_bat_count          0.0
acoustic_detection_count    0.0
nightly_visit_count         0.0
colony_estimate             0.0
comparison_species_count    0.0
observer_team               0.4
survey_quality              0.0
weather_interference        0.0
dtype: float64


- 20 valeurs manquantes, uniquement sur observer_team (0.4% du dataset).

### b. Hypothèse de départ

Plutôt que de remplir au hasard ou de supprimer les lignes, je pose une hypothèse : "peut-être que chaque site a toujours la même équipe assignée"

In [15]:
# Voir les lignes où observer_team est manquant
missing_team = df[df['observer_team'].isna()]
display(missing_team)

,survey_id,site_code,survey_start,survey_end,survey_method,observation_hours,observed_bat_count,acoustic_detection_count,nightly_visit_count,colony_estimate,comparison_species_count,observer_team,survey_quality,weather_interference
687,BAT-0000688,KNT-CYP,2024-04-05T18:00:00,2024-04-12T00:00:00,acoustic_and_visual,29.1,32,46,21,33.6,12,NaN,acceptable,36.6
902,BAT-0000903,KNT-NNM,2022-09-05T18:00:00,2022-09-12T00:00:00,acoustic_and_visual,31.2,8,18,11,10.6,15,NaN,acceptable,24.8
1382,BAT-0001383,KNT-WUQ,2023-09-05T18:00:00,2023-09-12T00:00:00,fixed_acoustic_station,30.6,9,13,9,11.6,16,NaN,high,59.6
1413,BAT-0001414,KNT-DMJ,2019-10-05T18:00:00,2019-10-12T00:00:00,acoustic_and_visual,27.8,5,5,2,7.2,18,NaN,high,43.9
2280,BAT-0002281,AVL-ARU,2020-07-05T18:00:00,2020-07-12T00:00:00,fixed_acoustic_station,15.7,5,5,1,4.4,10,NaN,limited,47.9
2292,BAT-0002293,AVL-ARU,2021-07-05T18:00:00,2021-07-12T00:00:00,acoustic_and_visual,29.9,12,19,12,13.6,13,NaN,acceptable,50.6
2303,BAT-0002304,AVL-ARU,2022-06-05T18:00:00,2022-06-12T00:00:00,acoustic_transect,25.8,13,18,12,15.7,11,NaN,acceptable,41.6
2600,BAT-0002601,AVL-RNY,2021-03-05T18:00:00,2021-03-12T00:00:00,acoustic_and_visual,39.9,39,38,22,49.9,7,NaN,acceptable,0.0
2900,BAT-0002901,AVL-YAU,2020-03-05T18:00:00,2020-03-12T00:00:00,acoustic_transect,27.0,23,23,22,24.5,12,NaN,high,26.6
3404,BAT-0003405,AVL-FNM,2023-03-05T18:00:00,2023-03-12T00:00:00,acoustic_and_visual,33.8,11,19,8,10.6,10,NaN,high,5.5


### c. Vérification de l'hypothèse (preuve)

In [16]:
# Pour chaque site_code des 20 lignes à trous, voir si on connaît l'équipe habituelle de ce site
sites_a_verifier = missing_team['site_code'].unique()

for site in sites_a_verifier:
    equipes_connues = df[(df['site_code'] == site) & (df['observer_team'].notna())]['observer_team'].unique()
    print(site, "->", equipes_connues)

KNT-CYP -> ['TEAM-11-2']
KNT-NNM -> ['TEAM-11-5']
KNT-WUQ -> ['TEAM-11-1']
KNT-DMJ -> ['TEAM-11-2']
AVL-ARU -> ['TEAM-12-3']
AVL-RNY -> ['TEAM-12-2']
AVL-YAU -> ['TEAM-12-1']
AVL-FNM -> ['TEAM-12-2']
AVL-NMQ -> ['TEAM-12-3']
TMB-WYT -> ['TEAM-13-3']
TMB-GWP -> ['TEAM-13-4']
TMB-HQL -> ['TEAM-13-1']
TMB-XTD -> ['TEAM-13-2']
TMB-SXP -> ['TEAM-13-4']
TMB-WQL -> ['TEAM-13-5']
TMB-QCB -> ['TEAM-13-1']


→ Résultat : chaque site n'a qu'une seule équipe dans tout le reste du dataset. Mon hypothèse est confirmée par les données elles-mêmes, pas par supposition

In [17]:
# Inférer observer_team à partir de l'équipe habituelle du site_code
df['observer_team'] = df.groupby('site_code')['observer_team'].transform(
    lambda x: x.fillna(x.mode()[0]) if not x.mode().empty else x
)

# Vérification : il ne doit plus y avoir de manquants
print("Valeurs manquantes restantes dans observer_team :", df['observer_team'].isna().sum())

Valeurs manquantes restantes dans observer_team : 0


# 3. Doublons

### a- Doublons de lignes

In [18]:
print("Nombre de lignes dupliquées :", df.duplicated().sum())
display(df[df.duplicated(keep=False)].sort_values('survey_id'))

Nombre de lignes dupliquées : 2


,survey_id,site_code,survey_start,survey_end,survey_method,observation_hours,observed_bat_count,acoustic_detection_count,nightly_visit_count,colony_estimate,comparison_species_count,observer_team,survey_quality,weather_interference
3182,BAT-0003183,AVL-ABG,2024-03-05T18:00:00,2024-03-12T00:00:00,acoustic_and_visual,38.7,8,18,12,8.6,7,TEAM-12-4,high,7.3
4992,BAT-0003183,AVL-ABG,2024-03-05T18:00:00,2024-03-12T00:00:00,acoustic_and_visual,38.7,8,18,12,8.6,7,TEAM-12-4,high,7.3
3566,BAT-0003567,AVL-FNQ,2023-09-05T18:00:00,2023-09-12T00:00:00,acoustic_transect,28.9,13,4,5,14.8,8,TEAM-12-4,high,55.9
4993,BAT-0003567,AVL-FNQ,2023-09-05T18:00:00,2023-09-12T00:00:00,acoustic_transect,28.9,13,4,5,14.8,8,TEAM-12-4,high,55.9


In [19]:
# Supprimer les doublons stricts (garder la première occurrence)
df = df.drop_duplicates(keep='first')

# Vérification
print("Nombre de lignes après suppression :", df.shape[0])
print("Doublons restants :", df.duplicated().sum())

Nombre de lignes après suppression : 4992
Doublons restants : 0


In [20]:
# Vérifier si deux colonnes catégorielles ont toujours la même correspondance
df.groupby('site_code')['observer_team'].nunique()

site_code
AVL-ABG    1
AVL-ARU    1
AVL-BVM    1
AVL-BWK    1
AVL-DER    1
          ..
TMB-VZT    1
TMB-WQL    1
TMB-WST    1
TMB-WYT    1
TMB-XTD    1
Name: observer_team, Length: 64, dtype: int64

- doublons de colomne :
On observe une redondance parfaite entre site_code et observer_team, mais on choisit de conserver les deux colonnes car elles encodent des informations conceptuellement distinctes (localisation vs. équipe assignée), utile pour la traçabilité des données. La redondance ne sera reconsidérée que si un modèle prédictif nécessite de réduire la colinéarité 

On verra s'il y a d'autres colonnes en correlation ou en double juste apres

### b- Doublons de colonnes 

In [68]:
# 1. Vérifier si des noms de colonnes sont strictement identiques
noms_doublons = df.columns[df.columns.duplicated()]
print("Noms de colonnes en double :", noms_doublons.tolist())

# 2. Vérifier si le contenu de certaines colonnes est 100% identique
contenu_doublons = df.columns[df.T.duplicated()]
print("Colonnes avec un contenu dupliqué :", contenu_doublons.tolist())

Noms de colonnes en double : []
Colonnes avec un contenu dupliqué : []


# 4. Formats incohérents

### Les types de chaque colonnes :

In [22]:
print(df.dtypes)

survey_id                    object
site_code                    object
survey_start                 object
survey_end                   object
survey_method                object
observation_hours            object
observed_bat_count           object
acoustic_detection_count     object
nightly_visit_count          object
colony_estimate             float64
comparison_species_count      int64
observer_team                object
survey_quality               object
weather_interference         object
dtype: object


### - Vérifier le format de survey_id

In [59]:
# Vérifier que toutes les valeurs suivent le pattern BAT-XXXXXXX (7 chiffres)
pattern = r'^BAT-\d{7}$'
non_conformes = df[~df['survey_id'].astype(str).str.match(pattern)]
print("Format non conforme :", len(non_conformes))
# print("Doublons :", df['survey_id'].duplicated().sum())

Format non conforme : 0


### - Vérifier le format de site_code (catégorielle)

In [45]:
print("Valeurs uniques :", df['site_code'].nunique())
print(df['site_code'].str.extract(r'^([A-Z]+)-')[0].value_counts())
print("Formats structurels :", df['site_code'].str.replace(r'[A-Z0-9]+', 'X', regex=True).unique())
#print("Manquants :", df['site_code'].isna().sum())

Valeurs uniques : 64
0
KNT    1716
AVL    1716
TMB    1560
Name: count, dtype: int64
Formats structurels : ['X-X']


### - Vérifier le format survey_start / survey_end (dates) <--->

In [28]:
start_dt = pd.to_datetime(df['survey_start'], errors='coerce')
end_dt = pd.to_datetime(df['survey_end'], errors='coerce')

print("survey_start non convertibles :", start_dt.isna().sum() - df['survey_start'].isna().sum())
print("survey_end non convertibles :", end_dt.isna().sum() - df['survey_end'].isna().sum())
print("Lignes où end <= start :", (end_dt <= start_dt).sum())

# Voir les formats bruts qui posent problème
mask_bad = end_dt.isna() & df['survey_end'].notna()
print("\nExemples de survey_end non convertibles :")
display(df.loc[mask_bad, ['survey_id', 'survey_start', 'survey_end']].head(10))

survey_start non convertibles : 0
survey_end non convertibles : 10
Lignes où end <= start : 0

Exemples de survey_end non convertibles :


,survey_id,survey_start,survey_end
114,BAT-0000115,2022-01-05T18:00:00,12/01/2022 00:00:00
246,BAT-0000247,2020-01-05T18:00:00,12/01/2020 00:00:00
408,BAT-0000409,2020-07-05T18:00:00,2020/07/12 00:00:00
414,BAT-0000415,2021-01-05T18:00:00,2021/01/12 00:00:00
655,BAT-0000656,2021-08-05T18:00:00,2021/08/12 00:00:00
1200,BAT-0001201,2021-07-05T18:00:00,12/07/2021 00:00:00
1329,BAT-0001330,2019-04-05T18:00:00,12-Apr-2019 00:00:00
2408,BAT-0002409,2024-09-05T18:00:00,12-Sep-2024 00:00:00
3041,BAT-0003042,2025-06-05T18:00:00,12/06/2025 00:00:00
3667,BAT-0003668,2019-02-05T18:00:00,12-Feb-2019 00:00:00


- Solution

In [55]:
# 1. Convertir en datetime en gérant les formats mixtes
df['survey_end'] = pd.to_datetime(df['survey_end'], format='mixed', errors='coerce', dayfirst=True)
df['survey_start'] = pd.to_datetime(df['survey_start'], format='mixed', errors='coerce', dayfirst=True)
# Vérification
print("survey_end non convertibles :", df['survey_end'].isna().sum())
print("Lignes où end <= start :", (df['survey_end'] <= df['survey_start']).sum())
print("Type de la colonne survey_end :", df['survey_end'].dtype)
print("Type de la colonne survey_start :", df['survey_start'].dtype)

survey_end non convertibles : 0
Lignes où end <= start : 0
Type de la colonne survey_end : datetime64[ns]
Type de la colonne survey_start : datetime64[ns]


### - Vérifier le format survey_method (catégorielle) <--->

In [47]:
print(df['survey_method'].value_counts(dropna=False))
print("Manquants :", df['survey_method'].isna().sum())

survey_method
acoustic_and_visual       1742
acoustic_transect         1616
fixed_acoustic_station    1614
FIXED_ACOUSTIC_STATION       9
ACOUSTIC_TRANSECT            4
ACOUSTIC_AND_VISUAL          4
Fixed_Acoustic_Station       2
Acoustic_Transect            1
Name: count, dtype: int64
Manquants : 0


- Solution 

In [51]:
# Convertir toutes les valeurs de la colonne en minuscules
df['survey_method'] = df['survey_method'].str.lower()

# Vérification du nettoyage
print(df['survey_method'].value_counts(dropna=False))

survey_method
acoustic_and_visual       1746
fixed_acoustic_station    1625
acoustic_transect         1621
Name: count, dtype: int64


### - Vérifier le format observation_hours (numérique) <--->

In [56]:
non_num = df[pd.to_numeric(df['observation_hours'], errors='coerce').isna() & df['observation_hours'].notna()]
print("Valeurs non convertibles :", len(non_num))
print("Exemples :", non_num['observation_hours'].unique()[:15])
print("Manquants :", df['observation_hours'].isna().sum())

Valeurs non convertibles : 0
Exemples : []
Manquants : 0


- Solution

In [57]:
# 1. Convertir en texte, remplacer la virgule par un point
df['observation_hours'] = df['observation_hours'].astype(str).str.replace(',', '.')

# 2. Convertir la colonne en numérique (float)
df['observation_hours'] = pd.to_numeric(df['observation_hours'], errors='coerce')

# Vérification
print("Valeurs non convertibles (devenues nulles) :", df['observation_hours'].isna().sum())
print("Type de la colonne :", df['observation_hours'].dtype)

Valeurs non convertibles (devenues nulles) : 0
Type de la colonne : float64


### - Vérifier le format observed_bat_count (numérique) <--->

In [31]:
non_num = df[pd.to_numeric(df['observed_bat_count'], errors='coerce').isna() & df['observed_bat_count'].notna()]
print("Valeurs non convertibles :", len(non_num))
print("Exemples :", non_num['observed_bat_count'].unique()[:15])
print("Manquants :", df['observed_bat_count'].isna().sum())

Valeurs non convertibles : 25
Exemples : ['9 count' '11 count' '16 count' '18 count' '5 count' '10 count' '1 count'
 '4 count' '19 count' '17 count' '6 count' '30 count' '13 count' '7 count'
 '20 count']
Manquants : 0


- Solution 

In [63]:
# 1. Convertir en texte temporairement et supprimer " count" (attention à l'espace avant)
df['observed_bat_count'] = df['observed_bat_count'].astype(str).str.replace(' count', '', regex=False)

# 2. Convertir la colonne en numérique
df['observed_bat_count'] = pd.to_numeric(df['observed_bat_count'], errors='coerce')

# Vérification
print("Valeurs non convertibles :", df['observed_bat_count'].isna().sum())
print("Type de la colonne :", df['observed_bat_count'].dtype)

Valeurs non convertibles : 0
Type de la colonne : int64


### - Vérifier le format acoustic_detection_count (numérique) <--->

In [32]:
non_num = df[pd.to_numeric(df['acoustic_detection_count'], errors='coerce').isna() & df['acoustic_detection_count'].notna()]
print("Valeurs non convertibles :", len(non_num))
print("Exemples :", non_num['acoustic_detection_count'].unique()[:15])
print("Manquants :", df['acoustic_detection_count'].isna().sum())

Valeurs non convertibles : 25
Exemples : ['7 count' '52 count' '47 count' '13 count' '18 count' '23 count'
 '37 count' '14 count' '9 count' '3 count' '28 count' '25 count'
 '33 count' '21 count' '41 count']
Manquants : 0


- Solution

In [64]:
# Supprimer le texte " count" et convertir en numérique
df['acoustic_detection_count'] = df['acoustic_detection_count'].astype(str).str.replace(' count', '', regex=False)
df['acoustic_detection_count'] = pd.to_numeric(df['acoustic_detection_count'], errors='coerce')

# Vérification
print("Valeurs non convertibles :", df['acoustic_detection_count'].isna().sum())
print("Type de la colonne acoustic_detection_count :", df['acoustic_detection_count'].dtype)

Valeurs non convertibles : 0
Type de la colonne acoustic_detection_count : int64


### - Vérifier le format nightly_visit_count (numérique) <--->

In [33]:
non_num = df[pd.to_numeric(df['nightly_visit_count'], errors='coerce').isna() & df['nightly_visit_count'].notna()]
print("Valeurs non convertibles :", len(non_num))
print("Exemples :", non_num['nightly_visit_count'].unique()[:15])
print("Manquants :", df['nightly_visit_count'].isna().sum())

Valeurs non convertibles : 10
Exemples : ['7 count' '1 count' '5 count' '14 count' '9 count' '8 count' '16 count'
 '4 count']
Manquants : 0


- Solution :

In [65]:
# Nettoyage du suffixe et conversion en numérique
df['nightly_visit_count'] = df['nightly_visit_count'].astype(str).str.replace(' count', '', regex=False)
df['nightly_visit_count'] = pd.to_numeric(df['nightly_visit_count'], errors='coerce')

# Vérification
print("Valeurs non convertibles :", df['nightly_visit_count'].isna().sum())
print("Type de la colonne nightly_visit_count :", df['nightly_visit_count'].dtype)

Valeurs non convertibles : 0
Type de la colonne nightly_visit_count : int64


### - Vérifier le format colony_estimate (numérique — déjà en float64)

In [34]:
print("Type :", df['colony_estimate'].dtype)
print("min :", df['colony_estimate'].min(), "| max :", df['colony_estimate'].max())
print("Valeurs négatives :", (df['colony_estimate'] < 0).sum())
print("Manquants :", df['colony_estimate'].isna().sum())

Type : float64
min : 0.0 | max : 99.6
Valeurs négatives : 0
Manquants : 0


### - Vérifier le format comparison_species_count (numérique — déjà en int64)

In [35]:
print("Type :", df['comparison_species_count'].dtype)
print("min :", df['comparison_species_count'].min(), "| max :", df['comparison_species_count'].max())
print("Valeurs négatives :", (df['comparison_species_count'] < 0).sum())
print("Manquants :", df['comparison_species_count'].isna().sum())

Type : int64
min : 1 | max : 26
Valeurs négatives : 0
Manquants : 0


### - Vérifier le format de observer_team (catégorielle)

In [38]:
pattern_team = r'^TEAM-\d+-\d+$'
non_conformes_team = df[~df['observer_team'].astype(str).str.match(pattern_team)]
print("Format non conforme :", len(non_conformes_team))
print(df['observer_team'].value_counts(dropna=False))
print("Manquants :", df['observer_team'].isna().sum())

Format non conforme : 0
observer_team
TEAM-11-2    390
TEAM-11-3    390
TEAM-11-4    390
TEAM-12-1    390
TEAM-12-2    390
TEAM-12-5    312
TEAM-13-3    312
TEAM-13-4    312
TEAM-11-5    312
TEAM-12-3    312
TEAM-12-4    312
TEAM-13-1    312
TEAM-13-5    312
TEAM-13-2    312
TEAM-11-1    234
Name: count, dtype: int64
Manquants : 0


### - Vérifier le format survey_quality (catégorielle)

In [39]:
print(df['survey_quality'].value_counts(dropna=False))
print("Manquants :", df['survey_quality'].isna().sum())

survey_quality
high          2688
acceptable    2166
limited        138
Name: count, dtype: int64
Manquants : 0


### - Vérifier le format weather_interference (numérique) <--->

In [66]:
non_num = df[pd.to_numeric(df['weather_interference'], errors='coerce').isna() & df['weather_interference'].notna()]
print("Valeurs non convertibles :", len(non_num))
print("Exemples :", non_num['weather_interference'].unique()[:15])
print("Manquants :", df['weather_interference'].isna().sum())

Valeurs non convertibles : 8
Exemples : ['15,0' '42,4' '7,7' '34,1' '33,9' '58,2' '6,4' '52,7']
Manquants : 0


- Solution : 

In [67]:
# Remplacer la virgule par un point puis convertir en numérique
df['weather_interference'] = df['weather_interference'].astype(str).str.replace(',', '.')
df['weather_interference'] = pd.to_numeric(df['weather_interference'], errors='coerce')

# Vérification
print("Valeurs non convertibles :", df['weather_interference'].isna().sum())
print("Type de la colonne weather_interference :", df['weather_interference'].dtype)

Valeurs non convertibles : 0
Type de la colonne weather_interference : float64


# 5. Gestion des valeurs categorielles (colonnes catégorielles)

### a. Encodage des variables ordonnées

In [69]:
# 1. Encodage des variables ORDONNÉES (survey_quality)
# On respecte la hiérarchie en attribuant des chiffres
mapping_quality = {'limited': 1, 'acceptable': 2, 'high': 3}
df['survey_quality'] = df['survey_quality'].map(mapping_quality)

### b. Encodage des variables nominales (non ordonnées)

In [70]:
# 2. ONE-HOT ENCODING strict pour les variables NOMINALES (non ordonnées)
cols_nominales = ['survey_method', 'observer_team', 'site_code']

# L'option drop_first=False garantit qu'on fait bien un One-Hot Encoding (toutes les colonnes sont conservées)
# L'option dtype=int permet d'avoir des 0 et des 1
df = pd.get_dummies(df, columns=cols_nominales, drop_first=False, dtype=int)

### - Affichage : 

In [71]:
# Affichage des infos pour vérifier que toutes les nouvelles colonnes ont bien été créées
print("Nouvelles dimensions du dataset :", df.shape)
display(df.head())

Nouvelles dimensions du dataset : (4992, 93)


,survey_id,survey_start,survey_end,observation_hours,observed_bat_count,acoustic_detection_count,nightly_visit_count,colony_estimate,comparison_species_count,survey_quality,...,site_code_TMB-PEL,site_code_TMB-QCB,site_code_TMB-SKZ,site_code_TMB-SXP,site_code_TMB-UVF,site_code_TMB-VZT,site_code_TMB-WQL,site_code_TMB-WST,site_code_TMB-WYT,site_code_TMB-XTD
0,BAT-0000001,2019-01-05 18:00:00,2019-01-12,28.6,7,13,7,3.8,8,3,...,0,0,0,0,0,0,0,0,0,0
1,BAT-0000002,2019-02-05 18:00:00,2019-02-12,31.7,8,12,9,7.3,18,2,...,0,0,0,0,0,0,0,0,0,0
2,BAT-0000003,2019-03-05 18:00:00,2019-03-12,32.4,22,28,14,19.9,8,3,...,0,0,0,0,0,0,0,0,0,0
3,BAT-0000004,2019-04-05 18:00:00,2019-04-12,35.7,26,36,19,32.2,14,3,...,0,0,0,0,0,0,0,0,0,0
4,BAT-0000005,2019-05-05 18:00:00,2019-05-12,32.0,23,29,22,20.9,16,2,...,0,0,0,0,0,0,0,0,0,0


# 6. Outliers + valeurs manquantes

# 7. Histogramme

# 8. Corrélations